<img src="images/panel_logo.png" width="10%" align="right">


# Big data dashboarding with Panel

In this notebook, we are going to put our visualizations together into a dashboard. We will also introduce a new tool from the HoloViz suite, Panel - a high-level app and dashboarding solution for Python.

---

## Reconnect to Dask Cluster

In [1]:
!pip install dask-gateway


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.4 MB/s eta 0:00:00


In [2]:
import dask_gateway
import dask.dataframe as dd

In [3]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = Client(cluster)

# Hiển thị lần lượt cả 2
display(cluster)
display(client)

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:46705
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40045'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37855'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:38867 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:38867
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:44410
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:33177 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:33177
INFO:distributed.core:Starting established connection to tcp://127

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46705,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38867,Total threads: 1
Dashboard: http://127.0.0.1:44413/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:40045,


## Load the airline on-time performance dataset

In [4]:
columns = [
    'YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_CARRIER',
    'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
    'DEP_TIME', 'DEP_DELAY', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED',
    'CANCELLATION_CODE', 'DIVERTED', 'AIR_TIME', 'FLIGHTS', 'DISTANCE',
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY',
    'LATE_AIRCRAFT_DELAY', 'DIV_ARR_DELAY'
]

In [5]:
columns = [
    'Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline',
    'Tail_Number', 'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime',
    'DepTime', 'DepDelay', 'ArrTime', 'ArrDelay', 'Cancelled',
    'CancellationCode', 'Diverted', 'AirTime', 'Flights', 'Distance',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay',
    'LateAircraftDelay'
]

In [10]:
import dask
import dask.dataframe as dd
import requests
import zipfile
import io
import os

# =========================
# 1. Đóng cluster cũ
# =========================
try:
    client.close()
except:
    pass

try:
    cluster.close()
except:
    pass

# Dùng scheduler local của Dask
dask.config.set(scheduler="threads")


# =========================
# 2. Tải dữ liệu tháng 1/2022
# =========================
url = (
    "https://transtats.bts.gov/PREZIP/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "1987_present_2022_1.zip"
)

r = requests.get(url)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_file = [
        f for f in z.namelist()
        if f.endswith(".csv")
    ][0]

    z.extract(csv_file, "/content")

csv_path = os.path.join("/content", csv_file)

print("Đã tải file:")
print(csv_path)


# =========================
# 3. Các cột dùng trong bài 07
# =========================
columns = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "Tail_Number",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "AirTime",
    "Flights",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]


# =========================
# 4. Đọc bằng Dask
# =========================
flights = dd.read_csv(
    csv_path,
    usecols=columns,
    dtype={
        "CancellationCode": "object",
        "FlightDate": "string",
        "DepTime": "float64",
        "ArrTime": "float64",
        "DepDelay": "float64",
        "ArrDelay": "float64",
        "Distance": "float64"
    },
    assume_missing=True,
    blocksize="64MB"
)


# =========================
# 5. Kiểm tra dữ liệu
# =========================
print("Số partition:", flights.npartitions)

flights.head(5, npartitions=1)

INFO:distributed.scheduler:Remove client Client-507b92fe-b392-11f1-9f92-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:44422; closing.
INFO:distributed.scheduler:Remove client Client-507b92fe-b392-11f1-9f92-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-507b92fe-b392-11f1-9f92-0242ac1c000c
INFO:distributed.scheduler:Closing scheduler. Reason: unknown
INFO:distributed.scheduler:Scheduler closing all comms


Đã tải file:
/content/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv
Số partition: 3


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CancellationCode,Diverted,AirTime,Flights,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2022.0,1.0,14.0,5.0,2022-01-14,YX,N119HQ,4879.0,CMH,DCA,...,<NA>,0.0,63.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
1,2022.0,1.0,15.0,6.0,2022-01-15,YX,N122HQ,4879.0,CMH,DCA,...,<NA>,0.0,50.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
2,2022.0,1.0,16.0,7.0,2022-01-16,YX,N412YX,4879.0,CMH,DCA,...,<NA>,0.0,53.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
3,2022.0,1.0,17.0,1.0,2022-01-17,YX,N405YX,4879.0,CMH,DCA,...,<NA>,0.0,56.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
4,2022.0,1.0,18.0,2.0,2022-01-18,YX,N420YX,4879.0,CMH,DCA,...,<NA>,0.0,48.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN


## Load the latitude-longitude data for the airports

We have a file `prep/airports.csv` that has the lat/lon information.

In [11]:
!pip install hvplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 4.8 MB/s eta 0:00:00


In [1]:

!pip install jupyter_bokeh

In [2]:
import panel as pn
import pandas as pd

pn.extension()

In [3]:
import hvplot.dask
import hvplot.pandas

In [4]:
import warnings

warnings.filterwarnings('ignore') # Ignore some HoloviewsDeprecationWarning

In [8]:
from google.colab import files
import pandas as pd
import os

# 1. Chọn file airports.csv từ máy
uploaded = files.upload()

# 2. Lấy tên file vừa upload
filename = list(uploaded.keys())[0]

print("Đã upload:", filename)

# 3. Đọc dữ liệu sân bay
airports = pd.read_csv(
    filename,
    usecols=[
        "AIRPORT",
        "DISPLAY_AIRPORT_NAME",
        "LATITUDE",
        "LONGITUDE"
    ],
    dtype={
        "AIRPORT": "string",
        "DISPLAY_AIRPORT_NAME": "string",
        "LATITUDE": "float",
        "LONGITUDE": "float"
    }
).set_index("AIRPORT")

# 4. Xóa mã sân bay bị trùng
airports = airports[
    ~airports.index.duplicated(keep="last")
]

# 5. Xem 5 dòng đầu
airports.head()

Saving airports.csv to airports.csv
Đã upload: airports.csv


,DISPLAY_AIRPORT_NAME,LATITUDE,LONGITUDE
AIRPORT,,,
01A,Afognak Lake Airport,58.109444,-152.906667
03A,Bear Creek Mining Strip,65.548056,-161.071667
04A,Lik Mining Camp,68.083333,-163.166667
05A,Little Squaw Airport,67.570000,-148.183889
06A,Kizhuyak Bay,57.745278,-152.882778


Let's quickly plot the map of airports:

In [9]:
!pip install geoviews cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 82.3 MB/s eta 0:00:00


In [11]:
airports.reset_index().hvplot.scatter(
    x="LONGITUDE",
    y="LATITUDE",
    color="red",
    alpha=0.2,
    hover_cols=[
        "AIRPORT",
        "DISPLAY_AIRPORT_NAME"
    ],
    width=900,
    height=500,
    title="Airport Locations"
)

:Scatter   [LONGITUDE]   (LATITUDE,AIRPORT,DISPLAY_AIRPORT_NAME)

## Build a Panel dashboard

Panel has 3 methods for building out interactive dashboards:

- hvPlot `.interactive` turns any of your DataFrame processing pipelines into a dashboard (great if you want to explore a dataset!);
- Panel `.bind` binds your widgets with your interactive plot (great if you want to build an arbitrary app!);
- Param encapsulates your dashboard as self-contained classes (great if you want to build a complex codebase supporting both GUI and non-GUI usage).

For more details, read this excellent blog post: [3 ways to build a panel visualization dashboard](https://towardsdatascience.com/3-ways-to-build-a-panel-visualization-dashboard-6e14148f529d).

We will use the `hvplot.interactive` method here, but first, let's build a small `pandas` data pipeline:

In [13]:
import os
import io
import zipfile
import requests
import dask
import dask.dataframe as dd

# Dùng scheduler local ổn định trên Colab
dask.config.set(scheduler="threads")

# =========================
# 1. Kiểm tra / tải CSV
# =========================
csv_path = (
    "/content/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2022_1.csv"
)

if not os.path.exists(csv_path):

    url = (
        "https://transtats.bts.gov/PREZIP/"
        "On_Time_Reporting_Carrier_On_Time_Performance_"
        "1987_present_2022_1.zip"
    )

    r = requests.get(url)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_file = [
            f for f in z.namelist()
            if f.endswith(".csv")
        ][0]

        z.extract(csv_file, "/content")

        csv_path = os.path.join(
            "/content",
            csv_file
        )

# =========================
# 2. Tạo lại flights
# =========================
columns = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "DepDelay",
    "ArrDelay",
    "Cancelled",
    "Distance"
]

flights = dd.read_csv(
    csv_path,
    usecols=columns,
    dtype={
        "FlightDate": "string",
        "DepDelay": "float64",
        "ArrDelay": "float64",
        "Distance": "float64"
    },
    assume_missing=True
)

# =========================
# 3. Tạo pipeline
# =========================
pipeline = (
    flights[
        (flights["FlightDate"] >= "2022-01-01") &
        (flights["FlightDate"] <= "2022-01-31")
    ]
    .groupby("DayOfWeek")["ArrDelay"]
    .mean()
    .to_frame(name="ArrDelay - mean")
)

pipeline

,ArrDelay - mean
npartitions=1,
,float64
,...


In [14]:
pipeline

,ArrDelay - mean
npartitions=1,
,float64
,...


In [15]:
pipeline.compute()

,ArrDelay - mean
DayOfWeek,
1.0,5.309050
2.0,-0.815602
3.0,-2.250541
4.0,1.252889
5.0,6.129445
6.0,5.439896
7.0,10.517304


## Move from static pipeline to a dynamic version

In the data pipeline above, let's use variables to represent the quantities we want to select in our dashboard:

- `daterange` - start and end dates to filter by
- `groupby` - variable we wish to groupby
- `field` - data field we wish to plot
- `method` - statistic we wish to calculate (min, max, mean etc)

### Pick some Panel widgets for each of these variables

Further reading: [Panel documentation](https://panel.holoviz.org/reference/index.html#widgets)

**We're picking the `DateRangeSlider` widget for the `daterange` variable:**

In [16]:
import datetime as dt

daterange = pn.widgets.DateRangeSlider(
    name='Date Range Slider',
    start=dt.datetime(2003, 1, 1), end=dt.datetime(2022, 12, 31),
    value=(dt.datetime(2022, 1, 1), dt.datetime(2022, 12, 31)),
    step=24*3600*2*1000,
    bar_color = "green",
    width=800
)

In [17]:
daterange

DateRangeSlider(bar_color='green', end=datetime.datetime(2022, ..., label='Date Range Slider', name='Date Range Slider', start=datetime.datetime(2003, ..., step=172800000, value=(datetime.datetime(2022, ..., value_end=datetime.datetime(2022, ..., value_start=datetime.datetime(2022, ..., width=800)

In [18]:
daterange.value

(datetime.datetime(2022, 1, 1, 0, 0), datetime.datetime(2022, 12, 31, 0, 0))

**We're picking the `RadioButtonGroup` widget for the `groupby` variable:**

In [19]:
groupby = pn.widgets.RadioButtonGroup(
    name='Period',
    options=['Year', 'Month', 'DayofMonth', 'Reporting_Airline'],
    value='Month',
)
groupby

RadioButtonGroup(label='Period', name='Period', options=['Year', 'Month', ...], value='Month')

In [20]:
groupby.value

'Month'

### 💻 Your turn: Choose an appropriate widget for `field` and `method`

In [21]:
# Your code here. When ready, click on the three dots below for the solutions.
import panel as pn

pn.extension()

# Chọn phép tính
method = pn.widgets.Select(
    name="Method",
    options=["min", "max", "mean", "count"],
    value="mean"
)

# Chọn biến cần phân tích
field = pn.widgets.RadioButtonGroup(
    name="Field",
    options=["DepDelay", "ArrDelay"],
    value="DepDelay"
)

# Hiển thị 2 widget
pn.Column(
    field,
    method
)

Column
    [0] RadioButtonGroup(label='Field', name='Field', options=['DepDelay', 'ArrDelay'], value='DepDelay')
    [1] Select(label='Method', name='Method', options=['min', 'max', ...], value='mean')

**We're picking the `RadioButtonGroup` widget for the `groupby` variable:**

In [22]:
method = pn.widgets.Select(
    name='Method',
    options=['min', 'max', 'mean', 'count'],
    value='mean',
)
method

Select(label='Method', name='Method', options=['min', 'max', ...], value='mean')

**We're picking the `RadioButtonGroup` widget for the `groupby` variable:**

In [23]:
field = pn.widgets.RadioBoxGroup(
    name='Field',
    options=['DepDelay', 'ArrDelay'],
)
field

RadioBoxGroup(label='Field', name='Field', options=['DepDelay', 'ArrDelay'], value='DepDelay')

## Put everything together with `hvplot.interactive`

First, we need to make the DataFrame into an interactive DataFrame:

In [24]:
iflights = flights.interactive()

Then, we can make an interactive pipeline using our widgets as variables

For reference, here's our pipeline code from before:

```python
pipeline = (
    flights[
        (flights['FL_DATE'] > "2020") &
        (flights['FL_DATE'] <= "2021")
    ]
    .groupby('DAY_OF_WEEK')["ARR_DELAY"].agg(how="mean")
)
```

In [25]:
import dask.dataframe as dd

# Ép kiểu cột FlightDate sang datetime
flights['FlightDate'] = dd.to_datetime(flights['FlightDate'])

# Khởi tạo lại interactive dataframe sau khi đã ép kiểu
iflights = flights.interactive()

In [27]:
import dask.dataframe as dd
import panel as pn
import hvplot.dask

# 1. Chuyển FlightDate sang datetime64
# KHÔNG dùng .dt.date
flights["FlightDate"] = dd.to_datetime(
    flights["FlightDate"],
    errors="coerce"
)

# 2. Tạo lại interactive dataframe
# vì iflights cũ có thể đang giữ kiểu dữ liệu trước đó
iflights = flights.interactive()

# 3. Pipeline tương tác
ipipeline = (
    iflights[
        (iflights["FlightDate"] >= daterange.param.value_start) &
        (iflights["FlightDate"] <= daterange.param.value_end)
    ]
    .groupby(groupby)[field]
    .agg(how=method)
    .reset_index()
)

# 4. Biểu đồ tương tác
ipipeline.hvplot.bar(
    x=groupby,
    y="how",
    title="Thống kê thời gian trễ chuyến bay"
)

In [28]:

flights.columns.tolist()

['Year',
 'Month',
 'DayofMonth',
 'DayOfWeek',
 'FlightDate',
 'Reporting_Airline',
 'Origin',
 'Dest',
 'DepDelay',
 'ArrDelay',
 'Cancelled',
 'Distance']

We can now use the interactive pipeline object in several ways.

In [29]:
data_plot = ipipeline.hvplot()
data_plot

### 💻 Your turn: Create an interactive version of the airport map plot, but color the airports based on data values

In [30]:
# Your code here. When ready, click on the three dots below for the solutions.
flight_delays = (
    iflights[
        (iflights["FlightDate"] >= daterange.param.value_start) &
        (iflights["FlightDate"] <= daterange.param.value_end)
    ]
    .groupby("Origin")[field]
    .agg(how=method)
    .join(airports)
    .rename(columns={"how": "DelayValue"})
)

flight_delays

In [31]:
flight_delays = (
    iflights[
        (iflights['FlightDate'] > daterange.value[0]) &
        (iflights['FlightDate'] <= daterange.value[1])
    ]
    .groupby('Origin')[field]
    .agg(how=method)
    .join(airports)
    .rename(columns={"how": f"{field} - {method}"})
)

flight_delays

In [33]:
map_plot = flight_delays.hvplot.scatter(
    x="LONGITUDE",
    y="LATITUDE",
    c="DelayValue",
    cmap="viridis",
    alpha=0.7,
    size=70,
    hover_cols=[
        "Origin",
        "DISPLAY_AIRPORT_NAME",
        "DelayValue"
    ],
    width=900,
    height=500,
    xlim=(-180, -30),
    ylim=(0, 75),
    title="Airport Delay Map"
)

map_plot

## Panel's Row/Column Grid system

Panel also has a customizable template system that allows you to build apps that have a header, sidebar, main area and popup windows. For details, see: https://panel.holoviz.org/user_guide/Templates.html

Let's re-arrange the components of our dashboard with this system:

In [36]:
import pandas as pd
import panel as pn
import hvplot.pandas

pn.extension()

# =====================================================
# 1. Lấy các cột cần dùng và tính sang Pandas một lần
# =====================================================
pdf = flights[
    [
        "FlightDate",
        "Year",
        "Month",
        "DayofMonth",
        "Reporting_Airline",
        "Origin",
        "DepDelay",
        "ArrDelay"
    ]
].compute(scheduler="threads")

pdf["FlightDate"] = pd.to_datetime(
    pdf["FlightDate"],
    errors="coerce"
)

# Dữ liệu sân bay
airports_df = airports.reset_index()


# =====================================================
# 2. Hàm tạo biểu đồ thống kê
# =====================================================
@pn.depends(
    daterange.param.value,
    field.param.value,
    groupby.param.value,
    method.param.value
)
def data_view(date_range, field_value, groupby_value, method_value):

    start_date = pd.to_datetime(date_range[0])
    end_date = pd.to_datetime(date_range[1])

    df = pdf[
        (pdf["FlightDate"] >= start_date) &
        (pdf["FlightDate"] <= end_date)
    ]

    grouped = df.groupby(groupby_value)[field_value]

    if method_value == "mean":
        result = grouped.mean()
    elif method_value == "min":
        result = grouped.min()
    elif method_value == "max":
        result = grouped.max()
    else:
        result = grouped.count()

    result = result.reset_index(name="Value")

    return result.hvplot.bar(
        x=groupby_value,
        y="Value",
        height=350,
        width=850,
        title=f"{method_value} của {field_value} theo {groupby_value}"
    )


# =====================================================
# 3. Hàm tạo biểu đồ vị trí sân bay
# =====================================================
@pn.depends(
    daterange.param.value,
    field.param.value,
    method.param.value
)
def map_view(date_range, field_value, method_value):

    start_date = pd.to_datetime(date_range[0])
    end_date = pd.to_datetime(date_range[1])

    df = pdf[
        (pdf["FlightDate"] >= start_date) &
        (pdf["FlightDate"] <= end_date)
    ]

    grouped = df.groupby("Origin")[field_value]

    if method_value == "mean":
        result = grouped.mean()
    elif method_value == "min":
        result = grouped.min()
    elif method_value == "max":
        result = grouped.max()
    else:
        result = grouped.count()

    result = result.reset_index(name="DelayValue")

    result = result.merge(
        airports_df,
        left_on="Origin",
        right_on="AIRPORT",
        how="inner"
    )

    return result.hvplot.scatter(
        x="LONGITUDE",
        y="LATITUDE",
        c="DelayValue",
        cmap="viridis",
        size=70,
        alpha=0.7,
        hover_cols=[
            "Origin",
            "DISPLAY_AIRPORT_NAME",
            "DelayValue"
        ],
        width=850,
        height=450,
        xlim=(-180, -30),
        ylim=(0, 75),
        title="Airport Delay Map"
    )


# =====================================================
# 4. Dashboard hoàn chỉnh
# =====================================================
dashboard = pn.Column(
    "# Airline On-Time Performance Dashboard",
    daterange,
    pn.Row(
        field,
        pn.Column(
            groupby,
            method
        )
    ),
    data_view,
    map_view
)

dashboard

Column
    [0] Markdown(str)
    [1] DateRangeSlider(bar_color='green', end=datetime.datetime(2022, ..., label='Date Range Slider', name='Date Range Slider', start=datetime.datetime(2003, ..., step=172800000, value=(datetime.datetime(2022, ..., value_end=datetime.datetime(2022, ..., value_start=datetime.datetime(2022, ..., width=800)
    [2] Row
        [0] RadioBoxGroup(label='Field', name='Field', options=['DepDelay', 'ArrDelay'], value='DepDelay')
        [1] Column
            [0] RadioButtonGroup(label='Period', name='Period', options=['Year', 'Month', ...], value='Month')
            [1] Select(label='Method', name='Method', options=['min', 'max', ...], value='mean')
    [3] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [4] ParamFunction(function, _pane=HoloViews, defer_load=False)

Close your Dask cluster:

In [38]:
cluster.close(shutdown=True)

Hoàn thành bài 07


---

## Next →

[Big data application pipeline](./08-big-data-application-pipeline.ipynb)